In [1]:
!pip install -q transformers
#pip install 'transformers[torch]' CPU version of PyTorch
#pip install 'transformers[tf-cpu]' CPU Version of TensorFlow

In [2]:
!pip install -q huggingface_hub

In [3]:
!pip install -q tokenizers

In [4]:
!pip install -q torch

In [7]:
!pip install -q accelerate

In [8]:
!pip install -q chromadb

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

huggingface_api_key  = os.environ['HUGGINGFACE_KEY']

In [5]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

/opt/homebrew/anaconda3/envs/GenerativeAI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    device_map='cuda',
    cache_dir='./base_models',
    token=huggingface_api_key
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf",
                                         cache_dir='./base_models.',
                                         token=huggingface_api_key)

In [ ]:
def get_llama2_chat_response(prompt, max_new_tokens=100):
  inputs = tokenizer(prompt, return_tensors="pt").to(device)
  outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.0001)
  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return response

## Vector DB

In [ ]:
import chromadb

chroma_client = chromadb.Client()

In [ ]:
collection = chroma_client.create_collection(name="my_collection")

In [ ]:
collection.add(
    embeddings = [[1,2],[2,1]],
    documents=["This is document 1", "This is document 2"],
    # metadatas=[{"source": "my_source"}, {"source": "my_source"}],
    ids=["id1", "id2"]
)

In [ ]:
results = collection.query(
    query_embeddings = [[4,1]],
    n_results=1
)

In [ ]:
results

{'ids': [['id2']],
 'distances': [[4.0]],
 'metadatas': [[None]],
 'embeddings': None,
 'documents': [['This is document 2']],
 'uris': None,
 'data': None}

## Embedding

In [ ]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('paraphrase-MiniLM-L6-v2',
                                      cache_folder="./base_models"
                                     )

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.73k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sentences = ["The man is sitting on the tree", "The girl is eating an apple"]
embeddings = embedding_model.encode(sentences)
embeddings

array([[-0.21421541, -0.05434414,  0.1983245 ,  0.10691632,  0.28119728,
         0.32432988,  0.1671172 ,  0.20439652, -0.35982794,  0.11064141,
        -0.08583811,  0.4164258 , -0.36450154,  0.8514863 ,  0.11012026,
         0.92857635, -0.3677544 ,  0.16180651,  0.44544208,  0.44167224,
         0.23353025,  0.02829285,  0.05719928, -0.7051381 ,  0.05321509,
        -0.16388665,  0.05863423,  0.01218422,  0.49752522, -0.15215288,
         0.29764953, -0.0795155 ,  0.7024071 ,  0.42614853, -0.47372615,
        -0.28052774, -0.454651  , -0.31223452,  0.10630041,  0.10367017,
        -0.56606895,  0.16192514,  0.14529678,  0.7097149 , -0.49968305,
         0.2386753 , -0.05783366, -0.46789366, -0.18654984,  0.14146505,
        -0.11448386, -0.13624062,  0.07725897, -0.41289458,  0.56032455,
         0.3067137 ,  0.00458776, -0.04948824,  0.6024603 ,  0.25524247,
         0.00164492, -0.7260487 ,  0.34768093, -0.22945064,  0.00646367,
        -0.28977177, -0.36313847, -0.4536401 ,  0.0

## RAG (Retreival Augmented Generation)

In [ ]:
!pip install PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.4 MB/s eta 0:00:00


In [ ]:
import PyPDF2

In [ ]:
# creating a pdf reader object
reader = PyPDF2.PdfReader('integrated_en.pdf')

In [ ]:
print(len(reader.pages))

65


In [ ]:
# print the text of the first page
print(reader.pages[0].extract_text())

Integrated 
Report 2022
Integrated Report 2022
Fiscal year ended March 31, 2022


In [ ]:
pages = [page.extract_text() for page in reader.pages]

In [ ]:
document = '\n'.join(pages)

In [ ]:
def get_overlapped_chunks(textin, chunksize, overlapsize):
    return [textin[a:a+chunksize] for a in range(0,len(textin), chunksize-overlapsize)]


In [ ]:
chunks = get_overlapped_chunks(document, 2000, 100)

In [ ]:
len(chunks)

210

In [ ]:
chunks[0]

'Integrated \nReport 2022\nIntegrated Report 2022\nFiscal year ended March 31, 2022\nTOYOTA MOTOR CORPORATION INTEGRATED REPORTMessage from \nthe PresidentThe Source of Our \nValue Creation:\n What Makes Us ToyotaValue Creation Story: \nWorking toward the Mobility \nSociety of the FutureBusiness Foundations \nfor Value CreationCorporate Data\n5 Our Founding Spirit   The Toyoda Principles and Toyota Philosophy   Toyota Production System (TPS)   Toyota and Sports\nThe Toyoda Principles and Toyota Philosophy\nWith an eye toward “Producing Happiness \nfor All” - Selections from the Q&A session of \nToyota’s 1H/2Q financial results briefing\nIn 1935, five years after the passing of Sakichi Toyoda, the \nCompany had grown to more than 10,000 employees as the \nautomotive business ramped up. The Toyoda Principles were \ncompiled at this time to convey Sakichi’s teachings to all employees \nand provide guidelines for all aspects of their work. \n The top management of Toyota that took over fro

In [ ]:
chunks[1]

'r all aspects of their work. \n The top management of Toyota that took over from Kiichiro, \nSakichi’s son, further codified the Toyota Philosophy, encompassing \nToyota’s values, priorities, and strengths. This philosophy provided \nthe answer to the fundamental question, “What is Toyota?” as \na\xa0touchstone for the entire Group.\n The automotive industry is experiencing a once-in-a-century transfor -\nmation. In the same way that Toyota transitioned from loom maker to \nautomaker, we are now reinventing ourselves as a mobility company.\n To guide us as we push forward into the future amid an era of \nuncertainty, we have now created the Toyota Philosophy Cone, \na\xa0graphic representation of the Toyota Philosophy presented in \na\xa0shape that evokes both the spools of thread used in looms and \nthe traffic cones used to guide cars. \nToyota Philosophy Cone\nToyota’s basic principles \nToyoda Principles \nThe Toyoda Principles, or Five \nMain Principles of Toyoda, \nhave since be

## Embedding

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', cache_folder = 'base_models')
chunk_embeddings = embedding_model.encode(chunks)
chunk_embeddings.shape

(210, 768)

In [ ]:
collection = chroma_client.create_collection(name="rag_llama2-1")

In [ ]:
collection.add(
    embeddings = chunk_embeddings,
    documents=chunks,
    ids= [str(i) for i in range(len(chunks))]
)

## Retrieval

In [ ]:
def retrieve_vector_db(query, n_results=3):
  results = collection.query(
  query_embeddings = embedding_model.encode(query).tolist(),
  n_results=n_results
  )
  return results['documents']

In [ ]:
query = "Who is Toyoda?"
retrieved_results = retrieve_vector_db(query)

In [ ]:
retrieved_results[0]

['ation \n worsened further. As the Company’s business \nresults showed no sign of improvement, \n labor-management negotiations with the Toyota \nMotor Co., Ltd. labor union deteriorated into a \nprotracted dispute. During collective bargaining \nthat April, the Company made reconstruction \nproposals centered on job cuts that the labor \nunion could not accept, and the dispute \n continued for another month and a half until \na\xa0memorandum was finally signed in June. \n Accepting responsibility for the labor disputes, \nKiichiro Toyoda resigned as president of the \nCompany in May 1950. In March 1952, he agreed \nto make his much-awaited return to the position, \nbut, before he could do so, he passed away at \nthe age of 57. Nevertheless, his aspirations were \nkept alive by his colleagues, who persevered with \npurely home-grown technologies as other \nJapanese automakers were forming technology \nalliances with U.S. and European manufacturers. \nThese efforts led to the 1955 laun

In [ ]:
context = '\n\n'.join(retrieved_results[0])
print(context)

ation 
 worsened further. As the Company’s business 
results showed no sign of improvement, 
 labor-management negotiations with the Toyota 
Motor Co., Ltd. labor union deteriorated into a 
protracted dispute. During collective bargaining 
that April, the Company made reconstruction 
proposals centered on job cuts that the labor 
union could not accept, and the dispute 
 continued for another month and a half until 
a memorandum was finally signed in June. 
 Accepting responsibility for the labor disputes, 
Kiichiro Toyoda resigned as president of the 
Company in May 1950. In March 1952, he agreed 
to make his much-awaited return to the position, 
but, before he could do so, he passed away at 
the age of 57. Nevertheless, his aspirations were 
kept alive by his colleagues, who persevered with 
purely home-grown technologies as other 
Japanese automakers were forming technology 
alliances with U.S. and European manufacturers. 
These efforts led to the 1955 launch of the 
Toyopet Crown, 

## Augmented Generation

In [ ]:
prompt = f'''
[INST]
Give answer for the question strictly based on the context provided.

Question: {query}

Context : {context}
[/INST]
'''

In [ ]:
prompt

'\n[INST]\nGive answer for the question strictly based on the context provided.\n\nQuestion: Who is Toyoda?\n\nContext : ation \n worsened further. As the Company’s business \nresults showed no sign of improvement, \n labor-management negotiations with the Toyota \nMotor Co., Ltd. labor union deteriorated into a \nprotracted dispute. During collective bargaining \nthat April, the Company made reconstruction \nproposals centered on job cuts that the labor \nunion could not accept, and the dispute \n continued for another month and a half until \na\xa0memorandum was finally signed in June. \n Accepting responsibility for the labor disputes, \nKiichiro Toyoda resigned as president of the \nCompany in May 1950. In March 1952, he agreed \nto make his much-awaited return to the position, \nbut, before he could do so, he passed away at \nthe age of 57. Nevertheless, his aspirations were \nkept alive by his colleagues, who persevered with \npurely home-grown technologies as other \nJapanese au

In [ ]:
print(get_llama2_chat_response(prompt, max_new_tokens=500))


[INST]
Give answer for the question strictly based on the context provided.

Question: Who is Toyoda?

Context : ation 
 worsened further. As the Company’s business 
results showed no sign of improvement, 
 labor-management negotiations with the Toyota 
Motor Co., Ltd. labor union deteriorated into a 
protracted dispute. During collective bargaining 
that April, the Company made reconstruction 
proposals centered on job cuts that the labor 
union could not accept, and the dispute 
 continued for another month and a half until 
a memorandum was finally signed in June. 
 Accepting responsibility for the labor disputes, 
Kiichiro Toyoda resigned as president of the 
Company in May 1950. In March 1952, he agreed 
to make his much-awaited return to the position, 
but, before he could do so, he passed away at 
the age of 57. Nevertheless, his aspirations were 
kept alive by his colleagues, who persevered with 
purely home-grown technologies as other 
Japanese automakers were forming technol

## RAG

In [ ]:
# 1. Retrieve Vector DB
query = "What is Toyoda's Principles?"

retrieved_results = retrieve_vector_db(query, n_results=5)
context = '\n\n'.join(retrieved_results[0])

# 2. Augment Context
prompt = f'''
[INST]
Give answer for the question strictly based on the context provided. Keep answers short and to the point.

Question: {query}

Context : {context}
[/INST]
'''

# 3. Pass to LLM for Generation
print(get_llama2_chat_response(prompt, max_new_tokens=2000))


[INST]
Give answer for the question strictly based on the context provided. Keep answers short and to the point.

Question: What is Toyoda's Principles?

Context : ive 
Main Principles of Toyoda, 
have since been handed 
down to every Toyota Group 
company and serve as 
guidelines for all employees.
Toyota’s mission since its foundation
Producing Happiness for All
Born into a family of poor farmers, Sakichi Toyoda built the bedrock of today’s Toyota by inventing the Toyoda Automatic Loom by 
himself. Abandoning the easy path left by his father, Kiicihro Toyoda took on the challenge of making cars. Many at the time said it 
was impossible.
 Their passion was carried on by those who worked with them, shaping the Toyota we know today. What they truly wanted to make 
was a sense of happiness for any customer who used their products, as well as happiness for every person involved in the work 
related to those products. The core of this aspiration was the idea of producing happiness for all

In [ ]:
# 1. Retrieve Vector DB
query = "What are units sales for Corolla in 2022?"

retrieved_results = retrieve_vector_db(query, n_results=5)
context = '\n\n'.join(retrieved_results[0])

# 2. Augment Context
prompt = f'''
[INST]
Give answer for the question strictly based on the context provided. Keep answers short and to the point.

Question: {query}

Context : {context}
[/INST]
'''

# 3. Pass to LLM for Generation
print(get_llama2_chat_response(prompt, max_new_tokens=2000))


[INST]
Give answer for the question strictly based on the context provided. Keep answers short and to the point.

Question: What are units sales for Corolla in 2022?

Context : 0/3 21/3 22/350%Figure 2  Changes in Break-even Vehicle Sales Volume*1
(FY2009 = 100%) Operating income basis
Message from the CFO: Changes in Profit Structure
FY2016*2FY2022*32,853.98,681 8,230 -451
+141.6-800.0
Forex/
Volume-1,000.0
Materials 
prices-200.0
Fixed costs +2,141.7
Proﬁt 
improvement, etc. 2,995.6 
¥120/USD 
¥133/Euro¥112/USD 
¥131/EuroConsolidated vehicle sales (thousands of vehicles)          
Operating income (billions of yen)Increased profit despite the negative effects of 
forex rates, sales volumes, and materials pricesFigure 3  FY2016 to FY2022: The Six Years Following the 
Adoption of the In-house Company System*1
Note: Data from ALG (a U.S. company). 
Percentages are the expected wholesale used 
car price 36 months in the future, divided by 
the new car retail price.
Increased market 
sha